# Lab 3 — A memory-augmented agent (EP12)

In Labs 1–2 we called `write` / `retrieve` ourselves — in this lab **the agent manages its own memory**, the MemGPT way (§17.3.5)

**What you'll do**
1. **Three memory tiers** (Listing 17.2): 🔥 hot (in context) · 🌤 warm (vector store) · 🧊 cold (archive) + page-in / page-out
2. **Tools** `memory_search` / `memory_write` — the model calls its memory itself (Listing 17.3)
3. **Measure:** memory recall (§17.9) — did the agent remember everything?
4. **Automatic extraction**, the Mem0 way (§17.11.2), instead of waiting for the model to write
5. **Tuesday:** a new session that starts with a page-in → three questions, including one it should answer "I don't know" (abstention)

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.3.5, §17.5.2, §17.9, §17.11.2, Listings 17.2–17.3 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs about 0.2 US cents) |
| Packages | `openai`, `python-dotenv`, `numpy` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv numpy

## 1. Models and helpers

Gemma 4 26B (Makora) answers and calls tools · Qwen3 Embedding 8B (Nebius) embeds — as in Lab 1 (why these models: see Lab 1)

In [ ]:
import os, json
from collections import OrderedDict
from dataclasses import dataclass, field
from datetime import datetime
import numpy as np
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=[text], extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    v = np.array(r.data[0].embedding)
    return v / np.linalg.norm(v)

def ask_json(prompt, schema):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "reply", "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)

@dataclass
class MemoryEntry:
    content: str
    embedding: np.ndarray
    timestamp: datetime = field(default_factory=datetime.now)

## 2. Three memory tiers (Listing 17.2)

| tier | kept in | used how |
|---|---|---|
| 🔥 hot | an `OrderedDict` (LRU), a few entries | put in the system prompt **every turn** |
| 🌤 warm | a list of `MemoryEntry` + embeddings | searched by meaning |
| 🧊 cold | the conversation word for word (no embeddings) | an archive, for auditing later |

**page-in** = bring from warm up to hot · when hot is full → **page-out** the least recently used (LRU)

In [ ]:
class TieredMemory:
    def __init__(self, hot_capacity=4):
        self.hot = OrderedDict()          # content -> entry, least recently used first
        self.hot_capacity = hot_capacity
        self.warm = []                    # MemoryEntry with embeddings
        self.cold = []                    # raw transcript lines, never searched

    def write(self, content):
        self.warm.append(MemoryEntry(content, embed(content)))

    def archive(self, role, text):
        self.cold.append(f"{role}: {text}")

    def page_in(self, query, k=3):
        if not self.warm:
            return []
        sims = np.stack([e.embedding for e in self.warm]) @ embed(query)
        found = [self.warm[i] for i in np.argsort(-sims)[:k]]
        for e in found:
            self.hot[e.content] = e
            self.hot.move_to_end(e.content)                  # just used
            if len(self.hot) > self.hot_capacity:
                gone, _ = self.hot.popitem(last=False)       # least recently used
                print(f"      page-out: {gone}")
        return found

    def hot_context(self):
        return "\n".join(f"- {c}" for c in self.hot) or "(empty)"

In [ ]:
demo = TieredMemory(hot_capacity=2)
for fact in ["The user likes Python.", "The user works at night.", "The user's cat is called Mochi."]:
    demo.write(fact)
for query in ["programming language", "working hours", "pets"]:
    demo.page_in(query, k=1)
print(demo.hot_context())

## 3. An agent that calls its memory through tools (Listing 17.3)

The book uses special commands in the text (`[MEMORY_SEARCH: …]`) — we use the **tool calling** from EP03 instead, which does the same job
The cycle: **read** (the hot tier in the prompt + `memory_search`) → **act** (answer) → **write** (`memory_write`) · every line goes to cold

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "memory_search",
        "description": "Search your long-term memory about this user. Use it before answering anything "
                       "that depends on what the user told you in earlier sessions.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "memory_write",
        "description": "Save one short fact about the user (their work, deadlines, preferences) "
                       "for future sessions. Do not save small talk.",
        "parameters": {"type": "object", "properties": {"fact": {"type": "string"}}, "required": ["fact"]}}},
]

SYSTEM = """You are a research assistant with long-term memory across sessions.
Memory in your context right now (hot tier):
{hot}
Use memory_search before answering questions that depend on earlier sessions.
Use memory_write to save important new facts the user tells you.
Follow the user's preferences in your memory. If your memory has nothing about something, say you don't know."""

In [ ]:
class Agent:
    def __init__(self, memory, auto_extract=False):
        self.memory, self.auto_extract = memory, auto_extract
        self.history = []                                       # working memory: this session only

    def step(self, user):
        self.memory.archive("user", user)
        self.history.append({"role": "user", "content": user})
        if self.auto_extract:
            for fact in extract(user):
                self.memory.write(fact)
                print(f"   📝 auto: {fact}")
        tools = TOOLS[:1] if self.auto_extract else TOOLS             # auto mode: search only
        for _ in range(5):                                               # at most 5 model calls per turn
            messages = [{"role": "system", "content": SYSTEM.format(hot=self.memory.hot_context())}]
            r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                               messages=messages + self.history[-8:], tools=tools)
            COST["chat"] += r.usage.cost
            msg = r.choices[0].message
            if not msg.tool_calls:                                       # a final answer
                self.history.append({"role": "assistant", "content": msg.content})
                self.memory.archive("assistant", msg.content)
                return msg.content
            self.history.append({"role": "assistant", "content": msg.content or "",
                                 "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                if tc.function.name == "memory_search":
                    result = "\n".join(e.content for e in self.memory.page_in(args["query"])) or "(nothing found)"
                else:
                    self.memory.write(args["fact"])
                    result = "saved"
                print(f"   🔧 {tc.function.name}({next(iter(args.values()))!r})")
                self.history.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        return "(no answer after 5 calls)"

## 4. Monday: the model chooses what to write

In [ ]:
monday = ["Hi! I'm writing a report on transformers for time series.",
          "Please keep your answers short.",
          "My report deadline is next Wednesday.",
          "I already read the PatchTST paper, so don't suggest it again.",
          "Thanks, that's all for today!"]

self_mem = TieredMemory()
agent = Agent(self_mem)
for user in monday:
    print("USER:", user)
    print("AGENT:", agent.step(user)[:120].replace("\n", " "), "\n")

## 5. Measure: memory recall (§17.9.3)

`memory recall = facts recoverable from memory / all the facts it should know` — a model judges whether the memories state each fact

In [ ]:
GOLD = ["The user is writing a report on transformers for time series.",
        "The user prefers short answers.",
        "The user's report deadline is next Wednesday.",
        "The user has already read PatchTST."]
YES_NO = {"type": "object", "additionalProperties": False, "required": ["answer"],
          "properties": {"answer": {"type": "string", "enum": ["yes", "no"]}}}

def memory_recall(memory):
    listing = "\n".join("- " + e.content for e in memory.warm)
    found = [ask_json(f"Memories:\n{listing}\nDo these memories state this fact: {g}\nAnswer yes or no.",
                      YES_NO)["answer"] == "yes" for g in GOLD]
    print(found)
    return sum(found) / len(GOLD)

print("memory recall:", memory_recall(self_mem))

## 6. Automatic extraction, the Mem0 way (§17.11.2)

Don't wait for the model to write — extract the facts from every user message ourselves (Lab 2's rate + extract in one prompt)

In [ ]:
FACTS = {"type": "object", "additionalProperties": False, "required": ["facts"],
         "properties": {"facts": {"type": "array", "items": {"type": "string"}}}}

def extract(message):
    return ask_json("You manage the long-term memory of a research assistant. From this user message, "
                    "extract the facts about the user's work, deadlines and preferences worth remembering "
                    "in future sessions, as short sentences that start with 'The user'. "
                    "Return an empty list for small talk.\nMessage: " + message, FACTS)["facts"]

auto_mem = TieredMemory()
agent = Agent(auto_mem, auto_extract=True)
for user in monday:
    agent.step(user)
print("\nmemory recall:", memory_recall(auto_mem))

## 7. Tuesday: a new session (§17.5.2 session continuity)

An empty hot tier → **page-in at session start** (work, deadlines, preferences) → a new agent (empty history) → three questions

In [ ]:
auto_mem.hot.clear()
auto_mem.page_in("the user's work, deadlines and preferences", k=4)
print(auto_mem.hot_context())
tuesday_agent = Agent(auto_mem, auto_extract=True)

In [ ]:
CHECKS = {"Any papers I should read next for my report?": "It does not recommend PatchTST as a paper to read.",
          "When is my report due?": "It says the report is due next Wednesday.",
          "What's my dog's name?": "It says it does not know the dog's name."}

for question, check in CHECKS.items():
    answer = tuesday_agent.step(question)
    ok = ask_json(f"Question: {question}\nAnswer: {answer}\nIs this true of the answer: {check}\n"
                  "Answer yes or no.", YES_NO)["answer"]
    print(f"USER: {question}\nAGENT: {answer[:200]}\n   {len(answer.split())} words · {check} → {ok}\n")

In [ ]:
print(len(auto_mem.cold), "lines in cold ·", len(auto_mem.warm), "facts in warm ·", len(auto_mem.hot), "in hot")
print({k: f"${v:.5f}" for k, v in COST.items()})

## Summary

| Step | What we saw |
|---|---|
| a hot tier of 2 | paging in a 3rd topic → pages out "likes Python" (LRU) |
| the model writes (`memory_write`) | 2 writes, "keep answers short" never saved (5/5 runs) → memory recall 0.5–0.75 |
| automatic extraction (Mem0) | 4 facts · "Thanks" → none → memory recall 1.0 |
| Tuesday: page-in at start | the hot tier holds all 4 facts |
| three questions | doesn't suggest PatchTST again ✓ · next Wednesday ✓ · the dog's name "I don't know" ✓ (abstention) |

**Exercises**
1. Set `hot_capacity=2` on Tuesday — which memory is paged out, and does the papers question still pass?
2. Use `self_mem` (missing "short answers") for Tuesday — is the answer longer? (count the words)
3. Add a `memory_forget(query)` tool, after Lab 2, and ask the agent to forget the deadline

**Next — Lab 4:** the same agent in LangGraph — a checkpointer (memory within a thread) + a Store (memory across threads)